In [20]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load training data
train_df = pd.read_csv("../data/train.csv")
train_df = train_df.reset_index(drop=True)

# 2. Feature engineering

# Extract Title from Name
train_df['Title'] = train_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Simplify rare titles
train_df['Title'] = train_df['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'],
    'Rare'
)
train_df['Title'] = train_df['Title'].replace(['Mlle', 'Ms'], 'Miss')
train_df['Title'] = train_df['Title'].replace('Mme', 'Mrs')

# Fill missing Embarked with mode
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)

# Create family size
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1

# Ensure Age is numeric
train_df['Age'] = pd.to_numeric(train_df['Age'], errors='coerce')

# Fill missing Age with median by Title group
train_df['Age'] = train_df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

# Calculate median Age per Title for use in test set
title_age_median = train_df.groupby('Title')['Age'].median()

# 3. Drop unused columns
train_df = train_df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# 4. One-hot encode categorical variables including Embarked & Title
train_df = pd.get_dummies(train_df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

# 5. Split features and target
X = train_df.drop('Survived', axis=1)
y = train_df['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 7. Hyperparameter tuning with GridSearchCV
logreg = LogisticRegression(max_iter=1000)
param_grid = {
    'C': [0.1, 0.5, 1, 5, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear']
}
grid_search = GridSearchCV(logreg, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# 8. Evaluate on validation set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_val_scaled)

print(f"Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred))
print("Classification Report:\n", classification_report(y_val, y_pred))

# 9. Load and preprocess test data
test_df = pd.read_csv("../data/test.csv")
test_df = test_df.reset_index(drop=True)

# Feature engineering on test data
test_df['Title'] = test_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test_df['Title'] = test_df['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'],
    'Rare'
)
test_df['Title'] = test_df['Title'].replace(['Mlle', 'Ms'], 'Miss')
test_df['Title'] = test_df['Title'].replace('Mme', 'Mrs')

test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1

test_df['Age'] = pd.to_numeric(test_df['Age'], errors='coerce')

# Impute missing Age in test data by Title median from train set
test_df['Age'] = test_df.apply(
    lambda row: title_age_median[row['Title']] if pd.isnull(row['Age']) else row['Age'], axis=1
)

test_df['Fare'].fillna(train_df['Fare'].median(), inplace=True)

# Drop unused columns
test_df = test_df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# One-hot encode test data - note: no Embarked in test data, so exclude it here
test_df = pd.get_dummies(test_df, columns=['Sex', 'Title'], drop_first=True)

# Add any missing columns from train set with zeros
for col in X.columns:
    if col not in test_df.columns:
        test_df[col] = 0

# Reorder columns to match train features
test_df = test_df[X.columns]

# Scale test features
X_test_scaled = scaler.transform(test_df)

# 10. Predict on test set
test_pred = best_model.predict(X_test_scaled)

# 11. Save submission file
submission = pd.DataFrame({
    'PassengerId': pd.read_csv("../data/test.csv")['PassengerId'],
    'Survived': test_pred
})

submission.to_csv("../submission/submission.csv", index=False)
print(" submission.csv saved to ../submission/submission.csv")


<>:15: SyntaxWarning: invalid escape sequence '\.'
<>:83: SyntaxWarning: invalid escape sequence '\.'
<>:15: SyntaxWarning: invalid escape sequence '\.'
<>:83: SyntaxWarning: invalid escape sequence '\.'
C:\Users\User\AppData\Local\Temp\ipykernel_5940\970021397.py:15: SyntaxWarning: invalid escape sequence '\.'
  train_df['Title'] = train_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
C:\Users\User\AppData\Local\Temp\ipykernel_5940\970021397.py:83: SyntaxWarning: invalid escape sequence '\.'
  test_df['Title'] = test_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
C:\Users\User\AppData\Local\Temp\ipykernel_5940\970021397.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

Best params: {'C': 0.5, 'penalty': 'l2', 'solver': 'lbfgs'}
Best cross-validation accuracy: 0.8300
Validation Accuracy: 0.8101
Confusion Matrix:
 [[88 17]
 [17 57]]
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.84      0.84       105
           1       0.77      0.77      0.77        74

    accuracy                           0.81       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179

 submission.csv saved to ../submission/submission.csv


C:\Users\User\AppData\Local\Temp\ipykernel_5940\970021397.py:100: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df['Fare'].fillna(train_df['Fare'].median(), inplace=True)
